<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake_Started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install drake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 134.5 MB/s eta 0:00:00


In [4]:
import pydrake.all;
print(pydrake.__file__)

/usr/local/lib/python3.12/dist-packages/pydrake/__init__.py


In [6]:
builder = pydrake.systems.framework.DiagramBuilder()
plant, _ = pydrake.multibody.plant.AddMultibodyPlantSceneGraph(builder, 0.0)
pydrake.multibody.parsing.Parser(builder).AddModels(
  pydrake.common.FindResourceOrThrow(
      "drake/examples/pendulum/Pendulum.urdf"))
plant.Finalize()
diagram = builder.Build()
simulator = pydrake.systems.analysis.Simulator(diagram)

In [11]:
import numpy as np
from pydrake.systems.analysis import Simulator
from pydrake.systems.framework import DiagramBuilder
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph, CoulombFriction
from pydrake.geometry import Box
from pydrake.math import RigidTransform # Ensuring RigidTransform is imported at the top
from pydrake.multibody.tree import SpatialInertia, UnitInertia # Import for SpatialInertia

# 1. The Builder
# Drake uses a "Diagram" structure. We start with an empty whiteboard (builder).
builder = DiagramBuilder()

# 2. Create the Physics Engine (Plant) and the Eye (SceneGraph)
# We add them together because physics needs to "see" collisions.
# time_step=1e-3 means the simulation updates every 1 millisecond.
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=1e-3)

# --- Define the World ---

# 3. Add the Ground (A Static Body)
# We get the world's internal "frame" (origin 0,0,0)
world_frame = plant.world_frame()

# We don't need to "create" a body for the ground, we just attach shape to the world.
# Dimensions: 10x10 meters, 1 meter thick.
plant.RegisterVisualGeometry(
    plant.world_body(),
    X_BG=RigidTransform(), # Changed p_BoBq to X_BG and np.eye(4) to RigidTransform()
    shape=Box(10, 10, 1),
    name="ground_visual",
    diffuse_color=np.array([0.5, 0.5, 0.5, 1.0]) # Changed list to numpy array
)
plant.RegisterCollisionGeometry(
    plant.world_body(),
    X_BG=RigidTransform(), # Changed np.eye(4) to RigidTransform() and added X_BG keyword
    shape=Box(10, 10, 1),
    name="ground_collision",
    coulomb_friction=CoulombFriction(1.0, 1.0) # Added keyword coulomb_friction
)

# 4. Add the Falling Block (A Dynamic Body)
# Create a rigid body named "brick" with an inertia (mass) of 1kg.
# (Note: Defining explicit SpatialInertia is verbose, simplifying here for concept)

# Define mass and dimensions for the brick (consistent with the Box shape later)
brick_mass = 1.0
brick_x, brick_y, brick_z = 1.0, 1.0, 1.0

# Create UnitInertia for a solid box
unit_inertia_brick = UnitInertia.SolidBox(brick_x, brick_y, brick_z)

# Create SpatialInertia object for the brick (CoM at body origin)
spatial_inertia_brick = SpatialInertia(
    mass=brick_mass,
    p_PScm_E=np.array([0., 0., 0.]), # Corrected from p_CoM
    G_SP_E=unit_inertia_brick # Corrected from G_BBo_B
)

brick = plant.AddRigidBody("brick", spatial_inertia_brick)

# Add shape to the brick
plant.RegisterVisualGeometry(
    brick,
    X_BG=RigidTransform(), # Changed np.eye(4) to RigidTransform() and added X_BG keyword
    shape=Box(brick_x, brick_y, brick_z),
    name="brick_visual",
    diffuse_color=np.array([0.9, 0.1, 0.1, 1.0]) # Changed list to numpy array
)
plant.RegisterCollisionGeometry(
    brick,
    X_BG=RigidTransform(), # Changed np.eye(4) to RigidTransform() and added X_BG keyword
    shape=Box(brick_x, brick_y, brick_z),
    name="brick_collision",
    coulomb_friction=CoulombFriction(0.5, 0.5) # Added keyword coulomb_friction
)

# 5. Finalize
# This tells Drake: "I am done adding bodies. Please compile the math."
plant.Finalize()

# 6. Build the Diagram
diagram = builder.Build()

# --- Simulation Phase ---

# 7. Set up the Simulator
simulator = Simulator(diagram)
context = simulator.get_mutable_context()

# 8. Set Initial Conditions
# We need to access the specific memory storage (Context) for the plant
plant_context = diagram.GetMutableSubsystemContext(plant, context)

# Set the brick's position: z = 5 meters high
# The state vector is [position_x, pos_y, pos_z, rotation..., velocity...]
# It's easier to use the SetFreeBodyPose API:
plant.SetFreeBodyPose(plant_context, brick, RigidTransform([0, 0, 5]))

# 9. Run!
print("Starting simulation...")
simulator.AdvanceTo(2.0) # Run for 2 seconds
print("Done!")

# Check where the brick landed
final_pose = plant.GetFreeBodyPose(plant_context, brick)
print(f"Final Brick Height: {final_pose.translation()[2]:.3f} meters")

Starting simulation...
Done!
Final Brick Height: 1.000 meters
